# Flash Attention

A refresher on **FlashAttention** — the IO-aware, *exact* attention algorithm that made long-context
Transformers practical. It computes the same numbers as vanilla attention but never materializes the
full `N×N` attention matrix in slow GPU memory, so it's both **faster** and **linear in memory** with
sequence length.

**Domain:** LLM Inference, Training & Optimization  ·  **recommended addition**  ·  **runnable:** yes  ·  _cross-ref [KV-Cache & Paged Attention](./kv-cache.ipynb) and [vLLM](./vllm.ipynb)_

## 1. What & Why

Self-attention computes `softmax(QKᵀ / √d) · V`. The naive implementation builds the full attention
score matrix `S = QKᵀ` of shape `N×N` (N = sequence length), writes it to GPU **HBM** (the big, slow
DRAM), reads it back to apply softmax, writes the probabilities, then reads them again to multiply by
`V`. Two problems:

1. **Memory is O(N²).** A 8K-token sequence needs a 8192×8192 matrix *per head per layer* — gigabytes.
   This is what makes long context blow up your VRAM, even though the *output* is only `N×d`.
2. **It's memory-bandwidth bound, not compute bound.** The arithmetic is cheap relative to the
   billions of bytes shuffled between HBM and the compute cores. The GPU spends most of its time
   waiting on memory, so the math FLOPs are nearly free by comparison.

**FlashAttention** (Dao et al., 2022) is an *IO-aware* rewrite. It splits Q, K, V into blocks (tiles)
that fit in the GPU's tiny, fast **SRAM**, and computes attention block-by-block using an **online
(streaming) softmax** that keeps a running max and running normalizer. The full `N×N` matrix is never
stored — only small tiles ever live in HBM. The result is **numerically identical** to standard
attention (it's exact, not an approximation), uses **O(N) memory** instead of O(N²), and runs
**2–4× faster** because it slashes HBM traffic.

**Reach for it when:** you train or serve Transformers, especially at longer context — it's the default
attention in modern stacks (PyTorch SDPA, vLLM, HF Transformers) and you usually just *enable* it.
**Don't expect magic when:** you're on CPU (the win comes from the GPU memory hierarchy), your
sequences are tiny, or you need an exotic attention bias an older kernel doesn't support.

## 2. Mental Model

Think of the GPU memory hierarchy like a **kitchen**: HBM is the big walk-in fridge (huge, slow to
walk to), SRAM is the **countertop** right next to you (tiny, instant). Naive attention cooks the whole
meal by repeatedly running to the fridge for every ingredient and storing every intermediate dish back
in it. FlashAttention keeps a small working set on the counter and finishes each portion before moving
on — far fewer trips to the fridge.

The algorithmic trick that makes this possible is the **online softmax**. Normally softmax needs the
*whole* row at once (to subtract the max for stability and to divide by the total sum). FlashAttention
processes the row in **blocks** and maintains two running statistics — the **max so far** `m` and the
**normalizer so far** `ℓ` — rescaling the partial output whenever a new block raises the max:

```
for each block of keys:
    S      = Q · Kⱼᵀ · scale            # a thin N×B slab, NOT the full N×N
    m_new  = max(m, rowmax(S))          # update running max
    P      = exp(S − m_new)             # safe exponent
    α      = exp(m − m_new)             # how much to shrink what we already have
    ℓ      = ℓ·α + rowsum(P)            # rescale + extend the denominator
    O      = O·α + P · Vⱼ              # rescale + accumulate the output
    m      = m_new
O = O / ℓ                               # final normalize, once
```

The one sentence to remember: **FlashAttention turns the O(N²) "compute the whole matrix, store it,
read it back" pattern into an O(N) streaming pass that keeps a running softmax and never leaves fast
memory — same answer, a fraction of the memory traffic.**

## 3. Key Concepts

| Term | What it means |
|------|---------------|
| **HBM vs SRAM** | GPU has lots of slow **HBM** (e.g. 40–80 GB, ~2 TB/s) and a tiny amount of fast on-chip **SRAM** (~20 MB, ~19 TB/s). FlashAttention is designed around keeping the hot data in SRAM. |
| **IO-aware** | The algorithm is optimized to minimize **reads/writes between HBM and SRAM**, not FLOPs. Attention is memory-bound, so cutting IO is what buys the speedup. |
| **Tiling / blocking** | Q, K, V are split into blocks sized to fit in SRAM; attention is computed one (Q-block, K-block) tile at a time. |
| **Online / streaming softmax** | Compute softmax incrementally over blocks using a running **max `m`** and running **sum `ℓ`**, rescaling earlier partial results when a later block raises the max. Mathematically exact. |
| **Rescaling factor `α = exp(m_old − m_new)`** | When a new block has larger values, everything accumulated so far was normalized against a smaller max — multiply it by `α` to correct it. |
| **Exactness** | FlashAttention produces the *same* output (up to floating-point) as vanilla attention — unlike Linformer/Performer/sparse attention, which approximate. |
| **Recomputation (backward)** | The backward pass doesn't store the `N×N` matrix either; it **recomputes** attention tiles from the saved softmax stats (`m`, `ℓ`). Trades a little extra compute for huge memory savings. |
| **FlashAttention-2 / -3** | Successive kernel rewrites: FA-2 improves work partitioning across warps/threadblocks for ~2× over FA-1; FA-3 exploits Hopper (H100) async + FP8 for further gains. |
| **SDPA** | PyTorch's `scaled_dot_product_attention` — the high-level entry point that auto-dispatches to a FlashAttention kernel when shapes/dtype/hardware allow. |

## 4. Setup

The worked examples below are **pure NumPy** so they run anywhere (CPU, no GPU, no downloads) — they
implement the online-softmax / tiled-attention math directly so you can *see* the mechanism and verify
it matches naive attention bit-for-bit. The real FlashAttention kernels only pay off on a GPU, so the
PyTorch path is gated and degrades gracefully on CPU.

```bash
# High-level: PyTorch's SDPA dispatches to FlashAttention automatically on supported GPUs
pip install torch

# The standalone kernels (Ampere/Ada/Hopper GPUs, fp16/bf16):
pip install flash-attn --no-build-isolation
```

`flash-attn` is **CUDA-only** and needs a fairly recent GPU (SM 8.0+, i.e. A100/RTX 30-series and
newer). For most users the right move is to *not* install it directly — just call
`torch.nn.functional.scaled_dot_product_attention` and let PyTorch pick the Flash kernel.

In [1]:
# Environment probe — what's available in THIS kernel (no downloads, no GPU needed).
import importlib.util
import sys

import numpy as np


def have(mod: str) -> str:
    return "installed" if importlib.util.find_spec(mod) else "not installed"


print(f"python        : {sys.version.split()[0]}")
print(f"numpy         : {np.__version__}")
for m in ("torch", "flash_attn", "transformers"):
    print(f"{m:<14}: {have(m)}")

print("\nWorked examples 1 & 2 are pure-NumPy and run regardless of the above.")

python        : 3.13.7
numpy         : 2.5.0
torch         : installed
flash_attn    : not installed
transformers  : installed

Worked examples 1 & 2 are pure-NumPy and run regardless of the above.


## 5. Worked Examples

### Example 1 — the online (streaming) softmax

This is the mathematical heart of FlashAttention. A normal softmax needs the whole row at once: it
subtracts the row max (for numerical stability) and divides by the sum of exponentials. The **online**
version processes the row in blocks, carrying just two running scalars — the max `m` and the
normalizer `ℓ` — and still lands on the exact same max and denominator. That's what lets the real
kernel avoid ever holding a full attention row in memory.

In [2]:
rng = np.random.default_rng(0)
x = rng.standard_normal(12) * 3.0   # stand-in for one row of attention scores


def softmax_stats_naive(x):
    """The two numbers softmax needs: the max (for stability) and the exp-sum (denominator)."""
    m = x.max()
    l = np.exp(x - m).sum()
    return m, l


def softmax_stats_online(x, block=4):
    """Same two numbers, computed block-by-block with a running max/sum — never sees all of x at once."""
    m = -np.inf   # running max
    l = 0.0       # running sum of exp(x - m)
    for s in range(0, len(x), block):
        blk = x[s:s + block]
        m_new = max(m, blk.max())
        # rescale the old sum to the new max, then add this block's contribution
        l = l * np.exp(m - m_new) + np.exp(blk - m_new).sum()
        m = m_new
    return m, l


m_naive, l_naive = softmax_stats_naive(x)
m_online, l_online = softmax_stats_online(x)
print(f"max   : naive={m_naive:.6f}  online={m_online:.6f}")
print(f"denom : naive={l_naive:.6f}  online={l_online:.6f}")
print(f"match : {np.allclose([m_naive, l_naive], [m_online, l_online])}")

max   : naive=3.912000  online=3.912000
denom : naive=1.641144  online=1.641144
match : True


### Example 2 — tiled FlashAttention forward vs. naive attention

Now the whole thing. `attention_naive` materializes the full `N×N` score matrix (what we want to
avoid). `flash_attention` walks over **blocks of keys**, and for each block updates a running max `m`,
running normalizer `ℓ`, and running output `O` — rescaling `O` and `ℓ` by `α = exp(m_old − m_new)`
whenever a block raises the max. It only ever holds a thin `N×block_k` slab, yet the output matches
naive attention to floating-point precision.

In [3]:
def attention_naive(Q, K, V, scale):
    S = (Q @ K.T) * scale                       # full N x N score matrix (the expensive part)
    S = S - S.max(axis=-1, keepdims=True)       # stabilize
    P = np.exp(S)
    P = P / P.sum(axis=-1, keepdims=True)        # softmax over keys
    return P @ V


def flash_attention(Q, K, V, scale, block_k=16):
    N, d = Q.shape
    O = np.zeros((N, d))
    m = np.full((N, 1), -np.inf)                # running row max
    l = np.zeros((N, 1))                        # running row normalizer
    for s in range(0, K.shape[0], block_k):
        Kj, Vj = K[s:s + block_k], V[s:s + block_k]
        S = (Q @ Kj.T) * scale                  # only N x block_k — a slab, not the full matrix
        m_new = np.maximum(m, S.max(axis=-1, keepdims=True))
        P = np.exp(S - m_new)                    # N x block_k
        alpha = np.exp(m - m_new)                # rescale factor for what we've accumulated
        l = l * alpha + P.sum(axis=-1, keepdims=True)
        O = O * alpha + P @ Vj
        m = m_new
    return O / l                                # one final normalize


N, d, block_k = 128, 32, 16
Q = rng.standard_normal((N, d))
K = rng.standard_normal((N, d))
V = rng.standard_normal((N, d))
scale = 1.0 / np.sqrt(d)

out_naive = attention_naive(Q, K, V, scale)
out_flash = flash_attention(Q, K, V, scale, block_k=block_k)

print(f"outputs match : {np.allclose(out_naive, out_flash)}")
print(f"max abs diff  : {np.abs(out_naive - out_flash).max():.2e}")
print()
print(f"naive peak score matrix : {N}x{N} = {N * N:,} floats")
print(f"flash peak score slab   : {N}x{block_k} = {N * block_k:,} floats "
      f"({N * N / (N * block_k):.0f}x smaller; in HBM this is O(N) vs O(N^2))")

outputs match : True
max abs diff  : 5.55e-16

naive peak score matrix : 128x128 = 16,384 floats
flash peak score slab   : 128x16 = 2,048 floats (8x smaller; in HBM this is O(N) vs O(N^2))


### Example 3 — PyTorch `scaled_dot_product_attention` (auto-dispatches to Flash)

In practice you never hand-roll the kernel — you call `F.scaled_dot_product_attention`, and PyTorch
selects a FlashAttention backend when the dtype/shape/hardware support it. The cell below runs the
high-level API on CPU (math fallback) to show the call shape and verify correctness; the actual
**Flash kernel + speed benchmark** is gated behind `RUN_GPU_FLASH` and a CUDA check so the notebook
still executes top-to-bottom anywhere.

In [4]:
import os

if importlib.util.find_spec("torch"):
    import torch
    import torch.nn.functional as F

    torch.manual_seed(0)
    # SDPA expects (batch, heads, seq, head_dim)
    q = torch.randn(1, 4, N, d)
    k = torch.randn(1, 4, N, d)
    v = torch.randn(1, 4, N, d)

    out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    print("SDPA output shape :", tuple(out.shape))

    # Cross-check our NumPy flash_attention against PyTorch (single head, non-causal)
    qn, kn, vn = q[0, 0].numpy(), k[0, 0].numpy(), v[0, 0].numpy()
    ref = F.scaled_dot_product_attention(q[:, :1], k[:, :1], v[:, :1])[0, 0].numpy()
    mine = flash_attention(qn, kn, vn, scale=1.0 / np.sqrt(d))
    print("matches our NumPy :", np.allclose(ref, mine, atol=1e-5))

    if os.getenv("RUN_GPU_FLASH") and torch.cuda.is_available():
        from torch.nn.attention import SDPBackend, sdpa_kernel

        dev = "cuda"
        big = [t.to(dev, dtype=torch.float16) for t in (
            torch.randn(2, 16, 4096, 64), torch.randn(2, 16, 4096, 64), torch.randn(2, 16, 4096, 64))]
        with sdpa_kernel(SDPBackend.FLASH_ATTENTION):   # force the Flash kernel
            o = F.scaled_dot_product_attention(*big, is_causal=True)
        print("GPU Flash kernel output:", tuple(o.shape))
    else:
        print("\n(Set RUN_GPU_FLASH=1 on a CUDA GPU to force + benchmark the Flash kernel.)")
else:
    print("torch not installed. API shape:")
    print("  out = F.scaled_dot_product_attention(q, k, v, is_causal=True)")
    print("  # q,k,v: (batch, heads, seq, head_dim); fp16/bf16 on Ampere+ -> Flash kernel")

SDPA output shape : (1, 4, 128, 32)
matches our NumPy : True

(Set RUN_GPU_FLASH=1 on a CUDA GPU to force + benchmark the Flash kernel.)


## 6. Gotchas & Pitfalls

- **It's exact, not approximate.** A common misconception is that FlashAttention "approximates"
  attention to go fast. It doesn't — same outputs (to floating point). The speed comes from IO, not
  from dropping terms. Don't expect accuracy loss, and don't confuse it with linear/sparse attention.
- **The win is GPU-specific.** The speedup comes from the HBM↔SRAM hierarchy. On CPU there's no such
  payoff — our NumPy version is for *understanding*, not speed. Benchmark on the actual target hardware.
- **dtype matters.** The fast kernels want **fp16/bf16**. In fp32 PyTorch may fall back to the slower
  math path. For training, this pairs naturally with mixed precision.
- **Hardware floor.** `flash-attn` needs SM 8.0+ (A100, RTX 30xx, and newer). On older GPUs (V100/T4)
  SDPA uses the *memory-efficient* (xFormers-style) backend instead — still O(N) memory, slightly slower.
- **head_dim limits.** Flash kernels support head dims up to 256 (version-dependent). Unusual head
  sizes can silently route to the fallback backend.
- **Custom attention bias / masks.** Older Flash kernels only do dense or causal masking; arbitrary
  additive bias (e.g. ALiBi, custom relative position) may not be supported and forces a fallback.
  Check before assuming you got the Flash path — use `torch.backends.cuda.sdp_kernel` context/`sdpa_kernel`.
- **Non-contiguous / odd shapes.** Stride or alignment quirks can disqualify the Flash backend. If
  you're not getting the speedup, print which backend SDPA actually chose.
- **It doesn't shrink the KV cache.** FlashAttention reduces the *attention compute's* memory, not the
  stored keys/values during autoregressive decoding — that's a separate problem (see the KV-cache
  notebook). The two are complementary.

## 7. When to Use vs Alternatives

| Approach | Exact? | Memory | Speed | Notes |
|----------|--------|--------|-------|-------|
| **Vanilla attention** | ✅ | **O(N²)** | baseline | Simple; blows up VRAM at long context. Fine for short seqs / teaching. |
| **FlashAttention (1/2/3)** | ✅ | **O(N)** | **2–4×** | Default choice on supported GPUs. Exact, IO-aware. Needs fp16/bf16 + recent NVIDIA HW. |
| **Memory-efficient (xFormers)** | ✅ | O(N) | ~1.5–2× | SDPA's fallback when Flash isn't available (older GPUs, odd masks). Same memory win, a bit slower. |
| **Linear attention** (Performer, Linformer) | ❌ (approx) | O(N) | fast | Approximates softmax with kernels/low-rank; can lose quality. Use only when exact is too expensive. |
| **Sparse attention** (Longformer, BigBird) | ❌ (restricted) | O(N·w) | fast | Each token attends to a window/global set. Good for very long docs where full attention is overkill. |
| **Ring / context-parallel attention** | ✅ | O(N) / device | scales | Shards the sequence across GPUs (often built on Flash) for million-token context. |

**Rules of thumb:**
- Training or serving on a modern GPU → **use FlashAttention** (i.e., just call SDPA / enable
  `attn_implementation="flash_attention_2"` in Transformers). It's free accuracy-wise.
- Older GPU or unsupported mask → you'll get the **memory-efficient** backend automatically; still good.
- Context so long that even O(N) per device is too much → **ring/sequence-parallel** attention.
- Willing to trade accuracy for extreme length/speed → **linear or sparse** attention, but measure the
  quality hit first; for most workloads exact FlashAttention is the better default.

## 8. Resources

- **FlashAttention paper** — *Dao et al., "FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness"* (NeurIPS 2022): <https://arxiv.org/abs/2205.14135>
- **FlashAttention-2** — *Dao, "Faster Attention with Better Parallelism and Work Partitioning"*: <https://arxiv.org/abs/2307.08691>
- **FlashAttention-3** — *Shah et al.*, Hopper async + FP8: <https://arxiv.org/abs/2407.08608>
- **Official repo** (Dao-AILab/flash-attention), install + supported configs: <https://github.com/Dao-AILab/flash-attention>
- **PyTorch SDPA docs** — `scaled_dot_product_attention` and backend selection: <https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html>
- **Online softmax** — *Milakov & Gimelshein, "Online normalizer calculation for softmax"* (the streaming-softmax trick): <https://arxiv.org/abs/1805.02867>
- **HF Transformers — GPU inference / FlashAttention-2** integration: <https://huggingface.co/docs/transformers/en/perf_infer_gpu_one>

**Cross-refs in this library:** [KV-Cache & Paged Attention](./kv-cache.ipynb) (the complementary decode-time memory problem), [vLLM](./vllm.ipynb) (serving stack that uses Flash + paged attention).